# Lab 8: Momentum & Match Phases - Advanced Feature Engineering

**Student Name:** [Your Name]
**Roll Number:** [Your Roll Number]

### 🎯 Objective
Use a provided utility function to aggregate raw match data. Then, engineer 'phase-based' run rate features to test if they improve model performance compared to a simple baseline. This entire practical is designed to be completed in **45-60 minutes**.

### Part 1: Data Aggregation (10-15 mins)

#### Task 1: Understand and Use the Provided Utility Function
In a real project, you're often given tools to handle repetitive tasks. Below is a function `aggregate_cricket_data` that reads a list of JSON file paths, parses them, and returns a single DataFrame with over-by-over data for all matches.

Your task is to:
1. Define a list containing the file paths of all your JSON files (for now, it will just be one file).
2. Call the function with this list.
3. Inspect the head of the resulting DataFrame `df_long` to understand its structure.

In [ ]:
import pandas as pd
import json
import os

def aggregate_cricket_data(file_paths):
    """Reads multiple cricket JSON files and aggregates them into a single over-by-over DataFrame."""
    all_match_data = []
    for file_path in file_paths:
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        # Navigate to the relevant data
        match_id = data['doc'][0]['data']['score']['matchId']
        innings = data['doc'][0]['data']['score']['innings'][0] # First innings only
        final_score = innings['runs']
        worm_data = data['doc'][0]['data']['score']['wormAndManhattan']
        
        for over_data in worm_data[:20]: # First 20 overs
            stats = over_data['firstInnings'].split(',')
            if len(stats) == 4:
                all_match_data.append({
                    'match_id': match_id,
                    'over': over_data['overNumber'],
                    'cumulative_runs': int(stats[2]),
                    'cumulative_wickets': int(stats[3]),
                    'final_score': final_score
                })
    return pd.DataFrame(all_match_data)

# 1. Define your list of files (we'll just use our one file multiple times to simulate)
json_files = ['get_scorecard.json'] # In a real scenario: ['match1.json', 'match2.json', ...]

# 2. Call the function
df_long = aggregate_cricket_data(json_files)

# 3. Inspect the DataFrame
# Your code here to display the head of df_long

### Part 2: Feature Engineering & Modeling (35-45 mins)

#### Task 2: Create Phase-Based Run Rate Features
Now for the main task. Complete the function `calculate_phase_run_rates` below. It takes the DataFrame for a single match and should return a `pd.Series` with the run rate for each phase. A helper function `get_runs_at_over` is provided for you.

In [ ]:
def get_runs_at_over(df, over_num):
    """Helper to get cumulative runs at a specific over."""
    if over_num == 0: return 0
    row = df[df['over'] == over_num]
    return row['cumulative_runs'].iloc[0] if not row.empty else 0

def calculate_phase_run_rates(match_df):
    # Get runs at the end of each phase boundary
    runs_at_6 = get_runs_at_over(match_df, 6)
    runs_at_10 = get_runs_at_over(match_df, 10)
    runs_at_15 = get_runs_at_over(match_df, 15)
    runs_at_20 = get_runs_at_over(match_df, 20)
    
    features = {}
    # --- YOUR CODE HERE ---
    # Calculate the run rate for each phase and add it to the 'features' dictionary
    # Example for Powerplay:
    # features['rr_powerplay'] = runs_at_6 / 6
    # Do the same for 'rr_middle1', 'rr_middle2', and 'rr_death'
    
    
    # --- END OF YOUR CODE ---
    features['final_score'] = match_df['final_score'].iloc[0]
    return pd.Series(features)

# Apply the function to all matches using groupby()
# df_features = df_long.groupby('match_id').apply(calculate_phase_run_rates)


#### Task 3: Compare Model Performance
Quickly train and compare two `RandomForestClassifier` models using `cross_val_score`:
1. **Baseline Model:** Uses only the `rr_powerplay` feature.
2. **Enhanced Model:** Uses all four phase-based run rate features.

### deliverables
A working notebook that correctly engineers phase-based features and provides a printout comparing the performance of the baseline and enhanced models.